# 1 — Download, prepare, and audit the complete dataset

## Question

Can the complete GSE205335 dataset be reconstructed as one reusable AnnData
object with raw counts, sample metadata, clinical metadata, and published cell
annotations aligned correctly?

This notebook keeps **all 96,505 cells and 33,714 genes** so the project can be
extended later. It does not perform response analysis or model evaluation.

## 0. Reproducible source preparation

For a fresh checkout, run these commands once from a terminal:

```bash
bash scripts/download_phase1_data.sh
Rscript scripts/export_rds_sparse.R
Rscript scripts/prepare_clinical_metadata.R
python scripts/build_anndata.py
```

The cells below audit the resulting file rather than redownloading 3 GB every
time the notebook is opened.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures/demo"
TABLES = RESULTS / "tables"
EMBEDDINGS = RESULTS / "embeddings"
for directory in (FIGURES, TABLES, EMBEDDINGS):
    directory.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA / "processed/GSE205335_phase1_raw_counts.h5ad"
RANDOM_STATE = 0

assert RAW_PATH.exists(), (
    f"Missing {RAW_PATH}. Run the four preparation commands shown above."
)
adata = ad.read_h5ad(RAW_PATH, backed="r")
print(adata)

## 1. Structural and count-matrix checks

In [ ]:
assert adata.shape == (96_505, 33_714)
assert adata.obs_names.is_unique
assert adata.var_names.is_unique
for start in range(0, adata.n_obs, 10_000):
    assert adata.X[start:min(start + 10_000, adata.n_obs)].min() >= 0
required = ["Sample", "Patient", "Tissue origin", "Platform", "core.patient",
            "Response", "lineage.total", "lineage.sub", "celltype"]
assert not set(required) - set(adata.obs.columns)
print("Shape:", adata.shape)
print("Samples:", adata.obs["Sample"].nunique())
print("Patients:", adata.obs["Patient"].nunique())
print("Sparse/raw non-negative counts verified across all cells")

## 2. Dataset inventory and fixed demo cohort

In [ ]:
inventory = pd.DataFrame({
    "cells": adata.obs.groupby("Sample", observed=True).size(),
    "Patient": adata.obs.groupby("Sample", observed=True)["Patient"].first(),
    "Tissue": adata.obs.groupby("Sample", observed=True)["Tissue origin"].first(),
    "Platform": adata.obs.groupby("Sample", observed=True)["Platform"].first(),
    "Core": adata.obs.groupby("Sample", observed=True)["core.patient"].first(),
})
inventory.to_csv(TABLES / "dataset_sample_inventory.csv")
display(inventory)

demo_mask = adata.obs["core.patient"].astype(str).eq("Core") & adata.obs["Tissue origin"].astype(str).eq("Metastatic LN")
demo = adata.obs.loc[demo_mask].groupby("Sample", observed=True).agg(
    cells=("Sample", "size"), Patient=("Patient", "first"),
    Response=("Response", "first"), Platform=("Platform", "first"),
).sort_index()
assert demo["cells"].sum() == 29_614 and len(demo) == 10
demo.to_csv(TABLES / "demo_10sample_cohort.csv")
display(demo)
print("Demo cohort: 29,614 cells, 10 metastatic-LN samples")
adata.file.close()

## Output

- Complete reusable dataset: `data/processed/GSE205335_phase1_raw_counts.h5ad`
- Complete sample inventory: `results/tables/dataset_sample_inventory.csv`
- Fixed demo cohort: `results/tables/demo_10sample_cohort.csv`